[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/probability_statistics/07_joint_distributions_and_multivariate_normal/exercises.ipynb)

# Exercises — Module 07: Joint Distributions and the Multivariate Normal

20 fully solved problems in four tiers: L0 — Concept Checks (4), L1 — Foundations (6), L2 — Applications (AI/ML and Physics) (6), L3 — Challenge Proofs (4).

In [1]:
import numpy as np
from scipy.stats import chi2, norm, multivariate_normal

rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

## L0 — Concept Checks

### Problem L0.1 — Do Marginals Determine the Joint?

Construct two different joint distributions on $\{0,1\}^2$ that both have $\text{Bernoulli}(1/2)$ marginals, and state what extra information the joint carries.

**Intuition**

The marginals are the row and column sums of the table, and many different tables share the same margins — so the interior of the table holds information the edges cannot see.

**Solution**

**Joint A (independent).** $P(x,y) = 1/4$ for each of the four cells:

| | $Y=0$ | $Y=1$ |
|---|---|---|
| $X=0$ | $0.25$ | $0.25$ |
| $X=1$ | $0.25$ | $0.25$ |

**Joint B (perfectly coupled).** $P(0,0) = P(1,1) = 1/2$, $P(0,1) = P(1,0) = 0$:

| | $Y=0$ | $Y=1$ |
|---|---|---|
| $X=0$ | $0.5$ | $0$ |
| $X=1$ | $0$ | $0.5$ |

Both have row sums and column sums $(1/2, 1/2)$, so both marginals are $\text{Bernoulli}(1/2)$. Yet $\rho_A = 0$ while $\rho_B = 1$; in B, $Y$ is a deterministic function of $X$. A whole one-parameter family interpolates: $P(0,0) = P(1,1) = \frac{1+\rho}{4}$ for $\rho \in [-1,1]$.

The extra information is the **dependence structure**. Sklar's theorem makes this precise: any joint splits into marginals plus a copula, and the copula is a free choice. This is why risk models built by specifying marginals alone are underdetermined — and why the 2008 crisis is often described as a copula-misspecification failure, not a marginal-misspecification one.

$$
\boxed{\text{Identical marginals, } \rho = 0 \text{ vs } \rho = 1: \text{ marginals never determine the joint}}
$$

*Key takeaway:* Marginals fix the axes; dependence is an independent modeling decision that must be specified, estimated, and stress-tested separately.

In [2]:
PA = np.full((2, 2), 0.25)
PB = np.array([[0.5, 0.0], [0.0, 0.5]])
vals = np.array([0.0, 1.0])


def corr(P):
    EX = (P.sum(1) * vals).sum()
    EY = (P.sum(0) * vals).sum()
    EXY = (P * np.outer(vals, vals)).sum()
    sx = np.sqrt((P.sum(1) * vals ** 2).sum() - EX ** 2)
    sy = np.sqrt((P.sum(0) * vals ** 2).sum() - EY ** 2)
    return (EXY - EX * EY) / (sx * sy)


for name, P in (("A (independent)", PA), ("B (coupled)", PB)):
    print(f"{name:18s} marginals X {P.sum(1)}  Y {P.sum(0)}  rho = {corr(P):+.4f}")
assert np.allclose(PA.sum(1), 0.5) and np.allclose(PB.sum(1), 0.5)
assert abs(corr(PA)) < 1e-12 and abs(corr(PB) - 1.0) < 1e-12

A (independent)    marginals X [0.5 0.5]  Y [0.5 0.5]  rho = +0.0000
B (coupled)        marginals X [0.5 0.5]  Y [0.5 0.5]  rho = +1.0000


### Problem L0.2 — Factorization Is Not Enough — the Support Matters

Let $f(x,y) = 2$ on the triangle $0 \lt x \lt y \lt 1$. The density "factors" as $2 \times 1$. Are $X$ and $Y$ independent?

**Intuition**

A density that looks like a product can still be tied together by its support: if the region where $f \gt 0$ is not a rectangle, knowing one coordinate changes the range of the other.

**Solution**

No. Compute the marginals. For $X$, integrate over $y \in (x, 1)$:

$$
f_X(x) = \int_x^1 2\,dy = 2(1-x), \qquad 0 \lt x \lt 1.
$$

For $Y$, integrate over $x \in (0, y)$:

$$
f_Y(y) = \int_0^y 2\,dx = 2y, \qquad 0 \lt y \lt 1.
$$

Their product is $4y(1-x)$, which is not $2$ on the triangle (and is nonzero off it). So $f_{X,Y} \ne f_Xf_Y$ and the variables are dependent.

The failure is entirely the **support**: the constraint $x \lt y$ couples the variables. Knowing $Y = 0.1$ confines $X$ to $(0, 0.1)$, while knowing $Y = 0.9$ allows $X$ up to $0.9$. Formally,

$$
f_{X\mid Y}(x \mid y) = \frac{2}{2y} = \frac{1}{y} \quad\text{on } (0,y),
$$

i.e. $X \mid Y = y \sim \text{Unif}(0,y)$ — manifestly $y$-dependent.

**Correct criterion:** $X \perp Y$ iff $f(x,y) = g(x)h(y)$ **and** the support is a product set $A \times B$.

$$
\boxed{\text{Dependent: } f_X(x) = 2(1-x), \; f_Y(y) = 2y, \; f_Xf_Y \ne f_{X,Y}}
$$

*Key takeaway:* A triangular, circular, or otherwise non-rectangular support creates dependence no matter how simple the density formula looks.

In [3]:
import sympy as sp

x, y = sp.symbols("x y", positive=True)
fX = sp.integrate(2, (y, x, 1))
fY = sp.integrate(2, (x, 0, y))
print("f_X(x) =", sp.simplify(fX), "   f_Y(y) =", sp.simplify(fY))
print("f_X * f_Y =", sp.expand(fX * fY), " vs joint 2  -> equal?",
      sp.simplify(fX * fY - 2) == 0)
assert sp.simplify(fX - 2 * (1 - x)) == 0 and sp.simplify(fY - 2 * y) == 0

f_X(x) = 2 - 2*x    f_Y(y) = 2*y
f_X * f_Y = -4*x*y + 4*y  vs joint 2  -> equal? False


### Problem L0.3 — Marginally Normal but Not Jointly Normal

Let $X \sim \mathcal{N}(0,1)$, let $S$ be an independent random sign, and set $Y = SX$. Show both marginals are standard normal and $\mathrm{Cov}(X,Y) = 0$, yet $X$ and $Y$ are dependent.

**Intuition**

Multiplying by a random sign reflects the bell curve onto itself, so the shape survives untouched and the correlation is killed by symmetry — while the mass stays glued to two lines.

**Solution**

**Marginal of $Y$.** Condition on $S$:

$$
P(Y \le y) = \tfrac12 P(X \le y) + \tfrac12 P(-X \le y) = \tfrac12\Phi(y) + \tfrac12\Phi(y) = \Phi(y),
$$

using the symmetry $-X \overset{d}{=} X$. So $Y \sim \mathcal{N}(0,1)$.

**Zero covariance.** $S \perp X$ with $E[S] = 0$, so

$$
\mathrm{Cov}(X,Y) = E\left[X\cdot SX\right] = E[S]\,E\left[X^2\right] = 0 \times 1 = 0.
$$

**Dependence.** $\lvert Y \rvert = \lvert X \rvert$ with probability 1. So $P\left(\lvert Y \rvert \gt 2 \mid \lvert X \rvert \lt 1\right) = 0$, whereas the unconditional probability is $0.0455$. All the mass sits on the two lines $y = \pm x$, so the "joint density" is singular — there is no joint density at all.

**Not jointly normal.** Take the linear combination $X + Y = X(1+S)$, which equals $2X$ with probability $1/2$ and $0$ with probability $1/2$. Its law has an atom of mass $1/2$ at 0, and no normal law has atoms. Since Definition 3.6 requires *every* linear combination to be normal, the pair is not jointly Gaussian — which is exactly why "uncorrelated implies independent" does not apply.

$$
\boxed{X, Y \sim \mathcal{N}(0,1) \text{ marginally}, \; \rho = 0, \text{ yet } \lvert Y \rvert = \lvert X \rvert \text{ a.s.}}
$$

*Key takeaway:* Joint normality is a statement about all linear combinations; marginal bell curves prove nothing, and only joint normality licenses the correlation-to-independence step.

In [4]:
m = 500_000
Xs = rng.normal(size=m)
Ys = rng.choice([-1.0, 1.0], size=m) * Xs
print(f"sample corr(X, Y)            = {np.corrcoef(Xs, Ys)[0, 1]:+.5f}   (population 0)")
print(f"P(|Y| > 2)                   = {np.mean(np.abs(Ys) > 2):.5f}")
print(f"P(|Y| > 2 | |X| < 1)         = {np.mean(np.abs(Ys[np.abs(Xs) < 1]) > 2):.5f}")
print(f"max |  |Y| - |X|  |          = {np.abs(np.abs(Ys) - np.abs(Xs)).max():.2e}")
assert abs(np.corrcoef(Xs, Ys)[0, 1]) < 0.01

sample corr(X, Y)            = -0.00004   (population 0)
P(|Y| > 2)                   = 0.04555
P(|Y| > 2 | |X| < 1)         = 0.00000
max |  |Y| - |X|  |          = 0.00e+00


### Problem L0.4 — Where Does Graph Structure Live — $\Sigma$ or $\Sigma^{-1}$?

For a chain $X_1 \to X_2 \to X_3$ with precision matrix $\Theta$ having $\Theta_{13} = 0$, explain why $\Sigma_{13} \ne 0$, and state which matrix a graphical model should estimate.

**Intuition**

In a chain, the two ends talk only through the middle: cut the middle out by conditioning and they fall apart, but leave it in and they move together.

**Solution**

Take the concrete chain

$$
\Theta = \begin{pmatrix}1 & -0.5 & 0\\ -0.5 & 1.5 & -0.5\\ 0 & -0.5 & 1\end{pmatrix}.
$$

The zero at $\Theta_{13}$ encodes $X_1 \perp X_3 \mid X_2$ (Theorem 4.3): once the intermediary is known, the endpoints carry no further information about each other.

Inverting, $\det\Theta = 1(1.5 - 0.25) - (-0.5)(-0.5 - 0) = 1.25 - 0.25 = 1.0$, and the $(1,3)$ cofactor gives

$$
\Sigma_{13} = \frac{(-0.5)(-0.5)}{\det\Theta} = 0.25 \ne 0.
$$

Marginally $X_1$ and $X_3$ *are* correlated, because both are driven by $X_2$. This is not a contradiction: marginal independence and conditional independence are different assertions, and neither implies the other.

**Consequence for modeling.** A sparse *covariance* means "most pairs are marginally uncorrelated" — rarely true and rarely interesting. A sparse *precision* means "most pairs are conditionally independent given everything else" — exactly the notion of a graph edge. Hence Gaussian graphical models (graphical lasso, neighbourhood selection) put the $\ell_1$ penalty on $\Theta$, and the estimated edge strengths are the partial correlations $-\Theta_{ij}/\sqrt{\Theta_{ii}\Theta_{jj}}$.

$$
\boxed{\Theta_{13} = 0 \text{ (conditional independence) but } \Sigma_{13} = 0.25 \ne 0 \text{ (marginal correlation)}}
$$

*Key takeaway:* Estimate the precision matrix when you want structure; the covariance matrix confounds direct relationships with paths through intermediaries.

In [5]:
Theta = np.array([[1.0, -0.5, 0.0], [-0.5, 1.5, -0.5], [0.0, -0.5, 1.0]])
Sigma = np.linalg.inv(Theta)
print("det Theta =", round(np.linalg.det(Theta), 12))
print("Sigma =\n", Sigma)
print(f"Theta_13 = {Theta[0, 2]:.4f}   Sigma_13 = {Sigma[0, 2]:.4f}")
assert abs(Sigma[0, 2] - 0.25) < 1e-12 and Theta[0, 2] == 0.0

det Theta = 1.0
Sigma =
 [[1.25 0.5  0.25]
 [0.5  1.   0.5 ]
 [0.25 0.5  1.25]]
Theta_13 = 0.0000   Sigma_13 = 0.2500


## L1 — Foundations

### Problem L1.1 — Marginals, Conditionals, and Independence From a Joint Density

Let $f(x,y) = c\left(x + y\right)$ on $[0,1]^2$. Find $c$, both marginals, $f_{Y \mid X}$, $E[Y \mid X = x]$, and decide independence.

**Intuition**

Everything about a pair of variables is already contained in the joint density; marginals are integrals of it, conditionals are normalised slices of it, and independence is the question of whether it factors.

**Solution**

**Normalization.**

$$
\int_0^1\!\!\int_0^1 c(x+y)\,dx\,dy = c\left(\frac{1}{2}+\frac{1}{2}\right) = c \implies c = 1.
$$

**Marginals.**

$$
f_X(x) = \int_0^1 (x+y)\,dy = x + \frac{1}{2}, \qquad f_Y(y) = y + \frac{1}{2}, \qquad 0 \le x, y \le 1.
$$

**Conditional.**

$$
f_{Y\mid X}(y \mid x) = \frac{x+y}{x+\tfrac12}, \qquad 0 \le y \le 1.
$$

It depends on $x$, so $X$ and $Y$ are **dependent** — confirmed by $f_Xf_Y = \left(x+\tfrac12\right)\left(y+\tfrac12\right) = xy + \tfrac{x+y}{2}+\tfrac14 \ne x+y$.

**Conditional mean.**

$$
E[Y \mid X = x] = \int_0^1 y\,\frac{x+y}{x+\frac12}\,dy = \frac{\frac{x}{2}+\frac{1}{3}}{x+\frac12} = \frac{3x+2}{6x+3}.
$$

At $x = 0$ this is $2/3$; at $x = 1$ it is $5/9 = 0.556$ — a *decreasing* function, so larger $X$ pushes $Y$ down. Consistent with a negative correlation: $E[XY] = \int\int xy(x+y) = \frac13\cdot\frac12+\frac12\cdot\frac13 = \frac13$, while $E[X] = E[Y] = \int_0^1 x(x+\tfrac12)dx = \tfrac{7}{12}$, so $\mathrm{Cov} = \frac13 - \frac{49}{144} = -\frac{1}{144}$.

$$
\boxed{c=1, \; f_X(x) = x+\tfrac12, \; E[Y\mid X=x] = \frac{3x+2}{6x+3}, \; \mathrm{Cov}(X,Y) = -\tfrac{1}{144}}
$$

*Key takeaway:* The whole toolkit — normalize, integrate for marginals, divide for conditionals, check factorization — runs mechanically off a single joint density.

In [6]:
import sympy as sp

x, y, c = sp.symbols("x y c")
f = c * (x + y)
c_val = sp.solve(sp.integrate(f, (x, 0, 1), (y, 0, 1)) - 1, c)[0]
f = f.subs(c, c_val)
fX = sp.integrate(f, (y, 0, 1))
EYgX = sp.simplify(sp.integrate(y * f / fX, (y, 0, 1)))
EX = sp.integrate(x * fX, (x, 0, 1))
EXY = sp.integrate(x * y * f, (x, 0, 1), (y, 0, 1))
cov = sp.simplify(EXY - EX ** 2)
print("c =", c_val, "  f_X(x) =", sp.simplify(fX), "  E[Y|X=x] =", EYgX)
print("E[X] =", EX, "  Cov(X,Y) =", cov, "=", float(cov))
assert c_val == 1 and cov == sp.Rational(-1, 144)
assert sp.simplify(EYgX - (3 * x + 2) / (6 * x + 3)) == 0

c = 1   f_X(x) = x + 1/2   E[Y|X=x] = (3*x + 2)/(3*(2*x + 1))
E[X] = 7/12   Cov(X,Y) = -1/144 = -0.006944444444444444


### Problem L1.2 — Building and Validating a Covariance Matrix

Decide whether $\Sigma = \begin{pmatrix}4 & 3\\ 3 & 2\end{pmatrix}$ is a valid covariance matrix, and compute $\mathrm{Var}\left(2X_1 - 3X_2\right)$ for $\Sigma = \begin{pmatrix}4 & 2\\ 2 & 9\end{pmatrix}$.

**Intuition**

A covariance matrix is legal exactly when no linear combination is assigned a negative variance, so testing validity and computing $\mathrm{Var}(\mathbf{a}^\top\mathbf{X})$ are the same quadratic-form calculation.

**Solution**

**Validity.** A covariance matrix must be symmetric positive semidefinite. Symmetry holds; check the eigenvalue signs via the determinant:

$$
\det\begin{pmatrix}4 & 3\\ 3 & 2\end{pmatrix} = 8 - 9 = -1 \lt 0,
$$

so one eigenvalue is negative and the matrix is **not** positive semidefinite — invalid. Equivalently, the implied correlation is $\rho = \frac{3}{\sqrt{4\cdot2}} = 1.06 \gt 1$, impossible. The probabilistic contradiction is explicit: taking $\mathbf{a} = (1, -1.5)^{\top}$,

$$
\mathbf{a}^{\top}\Sigma\mathbf{a} = 4(1) + 2(1)(-1.5)(3) + 2.25(2) = 4 - 9 + 4.5 = -0.5 \lt 0,
$$

which claims that the random variable $X_1 - 1.5X_2$ has negative variance.

**Linear combination.** For the valid $\Sigma = \begin{pmatrix}4&2\\2&9\end{pmatrix}$ and $\mathbf{a} = (2,-3)^{\top}$:

$$
\mathrm{Var}\left(\mathbf{a}^{\top}\mathbf{X}\right) = \mathbf{a}^{\top}\Sigma\mathbf{a} = 4(4) + 9(9) + 2(2)(-3)(2) = 16 + 81 - 24 = 73.
$$

Here $\rho = \frac{2}{\sqrt{36}} = \frac13$, and the positive correlation *reduces* the variance of the difference (the $-24$ term) relative to the uncorrelated value 97.

$$
\boxed{\text{First } \Sigma \text{ invalid } (\det \lt 0, \rho \gt 1); \quad \mathrm{Var}(2X_1-3X_2) = 73}
$$

*Key takeaway:* $\mathbf{a}^\top\Sigma\mathbf{a} = \mathrm{Var}\left(\mathbf{a}^\top\mathbf{X}\right) \ge 0$ *is* positive semidefiniteness — the algebraic condition and the probabilistic one are the same statement.

In [7]:
S_bad = np.array([[4.0, 3.0], [3.0, 2.0]])
S_ok = np.array([[4.0, 2.0], [2.0, 9.0]])
a = np.array([2.0, -3.0])
print(f"det(S_bad) = {np.linalg.det(S_bad):.4f}   eigenvalues {np.linalg.eigvalsh(S_bad)}")
print(f"a'S_bad a at a=(1,-1.5): {np.array([1.0, -1.5]) @ S_bad @ np.array([1.0, -1.5]):.4f}")
print(f"Var(2X1 - 3X2) = {a @ S_ok @ a:.4f}   hand 73")
assert np.linalg.eigvalsh(S_bad).min() < 0 and abs(a @ S_ok @ a - 73) < 1e-12

det(S_bad) = -1.0000   eigenvalues [-0.1623  6.1623]
a'S_bad a at a=(1,-1.5): -0.5000
Var(2X1 - 3X2) = 73.0000   hand 73


### Problem L1.3 — Conditioning in the Bivariate Normal

Height and weight are jointly normal with $\mu_H = 170$ cm, $\sigma_H = 10$, $\mu_W = 70$ kg, $\sigma_W = 12$, $\rho = 0.6$. Find the conditional law of weight given a height of 185 cm, and the 95% prediction interval.

**Intuition**

Gaussian conditioning is linear regression: the mean slides along the best-fit line by a fraction $\rho$ of the observed deviation, and the leftover spread is the same no matter where you stand.

**Solution**

Apply the bivariate conditioning formulas:

$$
E[W \mid H = h] = \mu_W + \rho\frac{\sigma_W}{\sigma_H}\left(h - \mu_H\right), \qquad \mathrm{Var}(W \mid H) = \sigma_W^2\left(1-\rho^2\right).
$$

**Mean.** The slope is $\rho\frac{\sigma_W}{\sigma_H} = 0.6 \times \frac{12}{10} = 0.72$ kg per cm, so

$$
E[W \mid H = 185] = 70 + 0.72(185-170) = 70 + 10.8 = 80.8\ \text{kg}.
$$

**Variance.**

$$
\mathrm{Var}(W \mid H) = 144\left(1 - 0.36\right) = 92.16, \qquad \mathrm{sd} = 9.6\ \text{kg},
$$

*independent of the observed height* — a distinctly Gaussian feature.

**Prediction interval.** $80.8 \pm 1.96(9.6) = 80.8 \pm 18.8$, i.e. $[62.0, 99.6]$ kg.

**Regression to the mean.** The person is $1.5$ standard deviations above mean height, yet the predicted weight is only $\frac{80.8-70}{12} = 0.9$ standard deviations above mean weight — the shrinkage factor is exactly $\rho$. Knowing height explains $\rho^2 = 36\%$ of weight variance, leaving 64% unexplained; that is the entire content of "correlation of 0.6".

$$
\boxed{W \mid H = 185 \sim \mathcal{N}(80.8,\ 92.16); \quad 95\% \text{ PI } = [62.0,\ 99.6]\ \text{kg}}
$$

*Key takeaway:* Gaussian conditioning moves the mean by $\rho$ standard deviations per standard deviation observed and shrinks the variance by the fixed factor $1-\rho^2$.

In [8]:
mu_H, sd_H, mu_W, sd_W, rho = 170.0, 10.0, 70.0, 12.0, 0.6
slope = rho * sd_W / sd_H
mean_c = mu_W + slope * (185 - mu_H)
var_c = sd_W ** 2 * (1 - rho ** 2)
lo, hi = mean_c - 1.96 * np.sqrt(var_c), mean_c + 1.96 * np.sqrt(var_c)
print(f"slope = {slope:.4f} kg/cm   E[W|H=185] = {mean_c:.4f}   Var = {var_c:.4f}")
print(f"95% prediction interval = [{lo:.2f}, {hi:.2f}]")
assert abs(mean_c - 80.8) < 1e-12 and abs(var_c - 92.16) < 1e-12

slope = 0.7200 kg/cm   E[W|H=185] = 80.8000   Var = 92.1600
95% prediction interval = [61.98, 99.62]


### Problem L1.4 — Affine Transformation of a Gaussian Vector

Let $\mathbf{X}\sim\mathcal{N}\left(\begin{pmatrix}1\\2\end{pmatrix},\begin{pmatrix}4&1\\1&9\end{pmatrix}\right)$ and $\mathbf{Y} = A\mathbf{X}+\mathbf{b}$ with $A = \begin{pmatrix}1&1\\1&-1\end{pmatrix}$, $\mathbf{b} = \begin{pmatrix}0\\3\end{pmatrix}$. Find the law of $\mathbf{Y}$ and determine whether its components are independent.

**Intuition**

An affine map stretches and rotates the ellipse of $\mathbf{X}$ without changing its Gaussian character, so the new law is fixed by pushing $\boldsymbol\mu$ and $\Sigma$ through the map.

**Solution**

**Mean.**

$$
E[\mathbf{Y}] = A\boldsymbol\mu + \mathbf{b} = \begin{pmatrix}1+2\\1-2\end{pmatrix}+\begin{pmatrix}0\\3\end{pmatrix} = \begin{pmatrix}3\\2\end{pmatrix}.
$$

**Covariance.** First $A\Sigma = \begin{pmatrix}1&1\\1&-1\end{pmatrix}\begin{pmatrix}4&1\\1&9\end{pmatrix} = \begin{pmatrix}5&10\\3&-8\end{pmatrix}$, then

$$
A\Sigma A^{\top} = \begin{pmatrix}5&10\\3&-8\end{pmatrix}\begin{pmatrix}1&1\\1&-1\end{pmatrix} = \begin{pmatrix}15&-5\\-5&11\end{pmatrix}.
$$

So $\mathbf{Y} \sim \mathcal{N}\left(\begin{pmatrix}3\\2\end{pmatrix}, \begin{pmatrix}15&-5\\-5&11\end{pmatrix}\right)$, with correlation $\rho = \frac{-5}{\sqrt{165}} = -0.389$.

**Independence.** Nonzero covariance means the components are dependent. But because $\mathbf{Y}$ *is* jointly Gaussian (affine images of Gaussians are Gaussian), independence would follow the instant the covariance vanished. Solving for when $Y_1 = X_1+X_2$ and $Y_2 = X_1 - X_2$ decorrelate: $\mathrm{Cov} = \sigma_1^2 - \sigma_2^2 = 4 - 9 = -5$, which is zero iff $\sigma_1 = \sigma_2$. The sum and difference of two jointly normal variables are independent exactly when they have equal variances — a fact used constantly in ANOVA-style orthogonal decompositions.

$$
\boxed{\mathbf{Y}\sim\mathcal{N}\left(\begin{pmatrix}3\\2\end{pmatrix},\begin{pmatrix}15&-5\\-5&11\end{pmatrix}\right); \text{ components dependent since } \sigma_1 \ne \sigma_2}
$$

*Key takeaway:* Affine closure reduces any linear-model question about Gaussians to two matrix products; the correlation of sum and difference is $\sigma_1^2-\sigma_2^2$.

In [9]:
mu = np.array([1.0, 2.0])
Sig = np.array([[4.0, 1.0], [1.0, 9.0]])
A = np.array([[1.0, 1.0], [1.0, -1.0]])
b = np.array([0.0, 3.0])
mu_Y, Sig_Y = A @ mu + b, A @ Sig @ A.T
print("mu_Y =", mu_Y, "  Sigma_Y =\n", Sig_Y)
print(f"rho_Y = {Sig_Y[0, 1] / np.sqrt(Sig_Y[0, 0] * Sig_Y[1, 1]):+.4f}   hand -0.389")
draws = rng.multivariate_normal(mu, Sig, size=400_000) @ A.T + b
print("Monte Carlo covariance =\n", np.cov(draws.T))
assert np.allclose(mu_Y, [3.0, 2.0]) and np.allclose(Sig_Y, [[15.0, -5.0], [-5.0, 11.0]])

mu_Y = [3. 2.]   Sigma_Y =
 [[15. -5.]
 [-5. 11.]]
rho_Y = -0.3892   hand -0.389
Monte Carlo covariance =
 [[14.9679 -4.9839]
 [-4.9839 10.9773]]


### Problem L1.5 — Sampling and Whitening via Cholesky

Given $\Sigma = \begin{pmatrix}4&2\\2&5\end{pmatrix}$ and $\boldsymbol\mu = (1,-1)^{\top}$, compute the Cholesky factor, transform $\mathbf{z} = (0.5, -1.2)^{\top}$ into a sample, and compute the Mahalanobis distance of $\mathbf{x} = (3, 1)^{\top}$.

**Intuition**

The Cholesky factor $L$ is the matrix that turns a round cloud of independent noise into the ellipse of $\Sigma$; running it forwards samples, running it backwards measures distance in the ellipse's own units.

**Solution**

**Cholesky.** Solve $LL^{\top} = \Sigma$ with $L$ lower triangular:

$$
L_{11} = \sqrt{4} = 2, \qquad L_{21} = \frac{2}{L_{11}} = 1, \qquad L_{22} = \sqrt{5 - 1^2} = 2, \qquad L = \begin{pmatrix}2&0\\1&2\end{pmatrix}.
$$

**Sampling.**

$$
\mathbf{x} = \boldsymbol\mu + L\mathbf{z} = \begin{pmatrix}1\\-1\end{pmatrix} + \begin{pmatrix}2(0.5)\\ 1(0.5)+2(-1.2)\end{pmatrix} = \begin{pmatrix}1\\-1\end{pmatrix}+\begin{pmatrix}1.0\\-1.9\end{pmatrix} = \begin{pmatrix}2.0\\-2.9\end{pmatrix}.
$$

**Mahalanobis distance of $(3,1)$.** Whiten by forward substitution on $L\mathbf{w} = \mathbf{x}-\boldsymbol\mu = (2, 2)^{\top}$:

$$
2w_1 = 2 \implies w_1 = 1, \qquad 1(1) + 2w_2 = 2 \implies w_2 = 0.5.
$$

$$
\Delta^2 = \lVert \mathbf{w}\rVert^2 = 1 + 0.25 = 1.25, \qquad \Delta = 1.118.
$$

**Checks.** Directly, $\Sigma^{-1} = \frac{1}{16}\begin{pmatrix}5&-2\\-2&4\end{pmatrix}$ and $(2,2)\Sigma^{-1}(2,2)^{\top} = \frac{1}{16}(20 - 8 - 8 + 16) = \frac{20}{16} = 1.25$. ✔ Also $\ln\det\Sigma = 2\left(\ln 2 + \ln 2\right) = 2\ln 4 = 2.7726$, matching $\ln 16$. ✔

Under $\chi^2_2$, $P\left(\Delta^2 \le 1.25\right) = 1 - e^{-0.625} = 0.465$ — an unremarkable point, sitting inside the 47% contour.

$$
\boxed{L = \begin{pmatrix}2&0\\1&2\end{pmatrix}, \quad \mathbf{x} = (2.0,\, -2.9)^{\top}, \quad \Delta^2 = 1.25}
$$

*Key takeaway:* One Cholesky factorization delivers sampling, whitening, Mahalanobis distances, and the log-determinant — never form $\Sigma^{-1}$ explicitly.

In [10]:
Sig = np.array([[4.0, 2.0], [2.0, 5.0]])
mu = np.array([1.0, -1.0])
L = np.linalg.cholesky(Sig)
z = np.array([0.5, -1.2])
xs = mu + L @ z
xq = np.array([3.0, 1.0])
w = np.linalg.solve(L, xq - mu)
d2 = w @ w
print("L =\n", L)
print("sample x = mu + L z =", xs)
print(f"Delta^2 = {d2:.6f}   direct {(xq - mu) @ np.linalg.inv(Sig) @ (xq - mu):.6f}   hand 1.25")
print(f"log det Sigma = {2 * np.log(np.diag(L)).sum():.6f}   ln 16 = {np.log(16):.6f}")
print(f"chi2_2 quantile of Delta^2 = {chi2.cdf(d2, df=2):.4f}   hand 0.465")
assert np.allclose(L, [[2.0, 0.0], [1.0, 2.0]]) and abs(d2 - 1.25) < 1e-12
assert np.allclose(xs, [2.0, -2.9])

L =
 [[2. 0.]
 [1. 2.]]
sample x = mu + L z = [ 2.  -2.9]
Delta^2 = 1.250000   direct 1.250000   hand 1.25
log det Sigma = 2.772589   ln 16 = 2.772589
chi2_2 quantile of Delta^2 = 0.4647   hand 0.465


### Problem L1.6 — Sum of Dependent Gaussians and Portfolio Variance

Assets have $\sigma_1 = 0.2$, $\sigma_2 = 0.3$, $\sigma_3 = 0.25$ with pairwise correlations $\rho_{12}=0.5$, $\rho_{13}=0.2$, $\rho_{23}=-0.3$. Compute the variance of the equally weighted portfolio and compare with the uncorrelated case.

**Intuition**

A portfolio's variance sums every entry of $\Sigma$, not just the diagonal, so the off-diagonal correlations decide how much the individual risks cancel rather than pile up.

**Solution**

Build the covariance matrix from $\Sigma_{ij} = \rho_{ij}\sigma_i\sigma_j$:

$$
\Sigma = \begin{pmatrix}0.0400 & 0.0300 & 0.0100\\ 0.0300 & 0.0900 & -0.0225\\ 0.0100 & -0.0225 & 0.0625\end{pmatrix}.
$$

With $\mathbf{w} = \left(\tfrac13,\tfrac13,\tfrac13\right)^{\top}$, the quadratic form is $\frac{1}{9}$ times the sum of all entries:

$$
\sum_{i,j}\Sigma_{ij} = \underbrace{(0.04+0.09+0.0625)}_{0.1925} + 2\underbrace{(0.03+0.01-0.0225)}_{0.0175} = 0.1925 + 0.0350 = 0.2275,
$$

$$
\mathrm{Var}\left(\mathbf{w}^{\top}\mathbf{R}\right) = \frac{0.2275}{9} = 0.02528, \qquad \mathrm{sd} = 0.1590 = 15.90\%.
$$

**Uncorrelated benchmark.** Dropping the off-diagonals gives $\frac{0.1925}{9} = 0.02139$, so $\mathrm{sd} = 14.62\%$. The correlations add $1.28$ percentage points of risk, driven mostly by the strong positive $\rho_{12}$, partly offset by the hedging $\rho_{23} = -0.3$.

**Naive benchmark.** Averaging the individual volatilities gives $\frac{0.2+0.3+0.25}{3} = 25\%$ — far above both. Diversification cuts risk from 25% to 15.9% here; with $n$ identical assets of correlation $\rho$ the limit is

$$
\mathrm{Var} = \frac{\sigma^2}{n} + \frac{n-1}{n}\rho\sigma^2 \xrightarrow[n\to\infty]{} \rho\sigma^2,
$$

so average correlation, not asset count, sets the floor on diversifiable risk.

$$
\boxed{\mathrm{sd} = 15.90\% \text{ (actual) vs } 14.62\% \text{ (if uncorrelated) vs } 25\% \text{ (no diversification)}}
$$

*Key takeaway:* Portfolio risk is $\mathbf{w}^\top\Sigma\mathbf{w}$, and the off-diagonal block — not the individual variances — determines how much diversification is available.

In [11]:
sd = np.array([0.2, 0.3, 0.25])
R = np.array([[1.0, 0.5, 0.2], [0.5, 1.0, -0.3], [0.2, -0.3, 1.0]])
Sig = np.outer(sd, sd) * R
w = np.full(3, 1 / 3)
var_p = w @ Sig @ w
var_u = w @ np.diag(np.diag(Sig)) @ w
print("Sigma =\n", Sig)
print(f"portfolio variance = {var_p:.6f}   sd = {np.sqrt(var_p):.4%}   hand 15.90%")
print(f"uncorrelated       = {var_u:.6f}   sd = {np.sqrt(var_u):.4%}   hand 14.62%")
print(f"no diversification (mean vol) = {sd.mean():.4%}")
assert abs(np.sqrt(var_p) - 0.1590) < 5e-5 and abs(np.sqrt(var_u) - 0.1462) < 5e-5

Sigma =
 [[ 0.04    0.03    0.01  ]
 [ 0.03    0.09   -0.0225]
 [ 0.01   -0.0225  0.0625]]
portfolio variance = 0.025278   sd = 15.8990%   hand 15.90%
uncorrelated       = 0.021389   sd = 14.6249%   hand 14.62%
no diversification (mean vol) = 25.0000%


## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Gaussian Process Regression Is One Conditioning Step

A GP with kernel $k(x,x') = \exp\left(-\frac{(x-x')^2}{2}\right)$ and noise $\sigma_n^2 = 0.01$ observes $y(0) = 1$ and $y(1) = 0$. Predict $f(0.5)$ with its posterior variance.

**Intuition**

A GP treats the unseen value and the observations as one Gaussian vector built from the kernel, so prediction is nothing more than slicing that vector with the conditioning formula.

**Solution**

**Assemble the joint.** The vector $\left(f(0.5), y(0), y(1)\right)$ is jointly Gaussian with mean zero. Kernel values: $k(0,1) = e^{-0.5} = 0.6065$, and $k(0.5, 0) = k(0.5,1) = e^{-0.125} = 0.8825$, $k(0.5,0.5) = 1$. Observations add noise on the diagonal:

$$
K = \begin{pmatrix}1.01 & 0.6065\\ 0.6065 & 1.01\end{pmatrix}, \qquad \mathbf{k}_* = \begin{pmatrix}0.8825\\ 0.8825\end{pmatrix}, \qquad k_{**} = 1.
$$

**Solve.** $\det K = 1.0201 - 0.3679 = 0.6522$, so

$$
K^{-1} = \frac{1}{0.6522}\begin{pmatrix}1.01 & -0.6065\\ -0.6065 & 1.01\end{pmatrix} = \begin{pmatrix}1.5486 & -0.9299\\ -0.9299 & 1.5486\end{pmatrix}.
$$

Then $\boldsymbol\alpha = K^{-1}\mathbf{y}$ with $\mathbf{y} = (1,0)^{\top}$ gives $\boldsymbol\alpha = (1.5486, -0.9299)^{\top}$.

**Posterior mean.**

$$
\mu_* = \mathbf{k}_*^{\top}\boldsymbol\alpha = 0.8825\left(1.5486 - 0.9299\right) = 0.8825 \times 0.6186 = 0.5459.
$$

Almost exactly the midpoint of the two observations, as symmetry demands.

**Posterior variance.** By the Schur complement, with $K^{-1}\mathbf{k}_* = 0.8825(0.6186, 0.6186)^\top = (0.5459, 0.5459)^{\top}$:

$$
\sigma_*^2 = k_{**} - \mathbf{k}_*^{\top}K^{-1}\mathbf{k}_* = 1 - 0.8825(0.5459 + 0.5459) = 1 - 0.9635 = 0.0365,
$$

so $\sigma_* = 0.191$ and the 95% band is $0.546 \pm 0.375$.

**Reading.** The posterior mean interpolates and the variance is small near the data and grows back to the prior value 1 far from it. Both numbers came from *exactly* the conditioning formula of Theorem 4.2.3 — a GP is nothing but a multivariate normal indexed by inputs.

$$
\boxed{f(0.5) \mid \mathbf{y} \sim \mathcal{N}(0.546,\ 0.0365), \quad 95\% \text{ CI} = [0.171,\ 0.921]}
$$

*Key takeaway:* GP regression requires no new theory — build the joint covariance from a kernel and apply Gaussian conditioning.

In [12]:
def k(u, v):
    return np.exp(-((u - v) ** 2) / 2)


Xtr = np.array([0.0, 1.0])
ytr = np.array([1.0, 0.0])
xs = 0.5
K = k(Xtr[:, None], Xtr[None, :]) + 0.01 * np.eye(2)
ks = k(xs, Xtr)
mu_s = ks @ np.linalg.solve(K, ytr)
var_s = 1.0 - ks @ np.linalg.solve(K, ks)
print("K =\n", K, "\nk_* =", ks, "  det K =", round(np.linalg.det(K), 6))
print(f"posterior mean     = {mu_s:.4f}   hand 0.546")
print(f"posterior variance = {var_s:.4f}   sd = {np.sqrt(var_s):.4f}   hand 0.0365")
print(f"95% CI = [{mu_s - 1.96 * np.sqrt(var_s):.3f}, {mu_s + 1.96 * np.sqrt(var_s):.3f}]")
assert abs(mu_s - 0.5459) < 1e-4 and abs(var_s - 0.03645) < 1e-5

K =
 [[1.01   0.6065]
 [0.6065 1.01  ]] 
k_* = [0.8825 0.8825]   det K = 0.652221
posterior mean     = 0.5459   hand 0.546
posterior variance = 0.0365   sd = 0.1909   hand 0.0365
95% CI = [0.172, 0.920]


### Problem L2.2 — The Kalman Update Is Gaussian Conditioning

A 1D state has prior $x \sim \mathcal{N}(10, 4)$ and a sensor gives $z = x + v$ with $v \sim \mathcal{N}(0, 1)$. Observing $z = 12$, derive the posterior and identify the Kalman gain.

**Intuition**

Prior and measurement are two noisy statements about the same quantity; conditioning weighs them by their precisions, and the gain is just the share of the total precision that the measurement owns.

**Solution**

**Joint law.** $(x, z)$ is jointly Gaussian since $z$ is an affine function of $(x, v)$:

$$
\begin{pmatrix}x\\z\end{pmatrix} \sim \mathcal{N}\left(\begin{pmatrix}10\\10\end{pmatrix}, \begin{pmatrix}4 & 4\\ 4 & 5\end{pmatrix}\right),
$$

because $\mathrm{Cov}(x,z) = \mathrm{Var}(x) = 4$ and $\mathrm{Var}(z) = 4 + 1 = 5$.

**Condition.**

$$
K = \Sigma_{xz}\Sigma_{zz}^{-1} = \frac{4}{5} = 0.8 \quad \text{(the Kalman gain)},
$$

$$
E[x \mid z=12] = 10 + 0.8(12-10) = 11.6, \qquad \mathrm{Var}(x \mid z) = 4 - \frac{16}{5} = 0.8.
$$

**Precision reading.** In precision units, $\frac{1}{0.8} = 1.25 = \frac{1}{4} + \frac{1}{1}$: **precisions add**, which is the cleanest statement of Bayesian fusion. The posterior mean is the precision-weighted average

$$
\frac{\frac14(10) + \frac11(12)}{\frac14+\frac11} = \frac{2.5+12}{1.25} = 11.6. \checkmark
$$

**Gain interpretation.** $K = \frac{\sigma_{\text{prior}}^2}{\sigma_{\text{prior}}^2+\sigma_{\text{noise}}^2}$ interpolates: with a perfect sensor ($\sigma_{\text{noise}}\to 0$) $K \to 1$ and the estimate jumps to the measurement; with a useless sensor $K \to 0$ and the prior is retained. And the posterior variance $0.8$ is smaller than *both* inputs — combining information always helps, and it does so by an amount known before the measurement arrives.

$$
\boxed{K = 0.8, \quad x \mid z = 12 \sim \mathcal{N}(11.6,\ 0.8), \quad \text{precisions add}}
$$

*Key takeaway:* The Kalman filter is Theorem 4.2 applied repeatedly — predict by affine closure, update by conditioning; nothing else is happening.

In [13]:
var_x, var_v, z_obs, mu_x = 4.0, 1.0, 12.0, 10.0
Sig = np.array([[var_x, var_x], [var_x, var_x + var_v]])
K = Sig[0, 1] / Sig[1, 1]
post_mean = mu_x + K * (z_obs - mu_x)
post_var = Sig[0, 0] - Sig[0, 1] ** 2 / Sig[1, 1]
print(f"Kalman gain K       = {K:.4f}   hand 0.8")
print(f"posterior mean      = {post_mean:.4f}   hand 11.6")
print(f"posterior variance  = {post_var:.4f}   hand 0.8")
print(f"precision check     : 1/{post_var:.1f} = {1 / post_var:.4f} = 1/4 + 1/1 = {0.25 + 1.0}")
assert abs(K - 0.8) < 1e-12 and abs(post_mean - 11.6) < 1e-12 and abs(post_var - 0.8) < 1e-12

Kalman gain K       = 0.8000   hand 0.8
posterior mean      = 11.6000   hand 11.6
posterior variance  = 0.8000   hand 0.8
precision check     : 1/0.8 = 1.2500 = 1/4 + 1/1 = 1.25


### Problem L2.3 — PCA as the Eigendecomposition of $\Sigma$

For $\Sigma = \begin{pmatrix}5 & 4\\ 4 & 5\end{pmatrix}$, find the principal components, the variance explained, and the 95% confidence ellipse axes.

**Intuition**

The covariance matrix draws an ellipse; its eigenvectors are the ellipse's axes and the eigenvalues are the variances along them, so PCA is simply reading off that shape.

**Solution**

**Eigendecomposition.** The characteristic equation is $(5-\lambda)^2 - 16 = 0$, so $5 - \lambda = \pm4$ and

$$
\lambda_1 = 9, \qquad \lambda_2 = 1.
$$

Eigenvectors: for $\lambda_1 = 9$, $(5-9)v_1 + 4v_2 = 0$ gives $\mathbf{u}_1 = \frac{1}{\sqrt2}(1,1)^{\top}$; orthogonality gives $\mathbf{u}_2 = \frac{1}{\sqrt2}(1,-1)^{\top}$.

**Variance explained.** $\mathrm{tr}\,\Sigma = 10 = \lambda_1+\lambda_2$, so PC1 carries $9/10 = 90\%$ and PC2 carries $10\%$. Projecting onto PC1 alone loses only 10% of the total variance while halving the dimension — the standard PCA trade.

**Interpretation.** PC1 is the "common mode" $\frac{X_1+X_2}{\sqrt2}$ with variance 9; PC2 is the "differential mode" $\frac{X_1-X_2}{\sqrt2}$ with variance 1. Strong positive correlation ($\rho = 0.8$) means the variables mostly move together, which is exactly what a dominant first eigenvalue reports.

**Confidence ellipse.** By Proof 5.6, $\Delta^2 \sim \chi^2_2$ and $\chi^2_{2,0.95} = 5.991$. The ellipse $\Delta^2 = 5.991$ has semi-axes

$$
a = \sqrt{\lambda_1 \times 5.991} = \sqrt{53.92} = 7.34 \text{ along } \mathbf{u}_1, \qquad b = \sqrt{\lambda_2\times5.991} = 2.45 \text{ along } \mathbf{u}_2,
$$

an elongated ellipse tilted at $45^\circ$ with aspect ratio $\sqrt{\lambda_1/\lambda_2} = 3$.

**Whitening contrast.** Whitening rescales both directions to unit variance, turning the ellipse into a circle — useful for distance-based methods, but it deliberately destroys the variance ordering that PCA exists to expose.

$$
\boxed{\lambda = (9, 1), \; \mathbf{u}_1 = \tfrac{1}{\sqrt2}(1,1)^\top \text{ explains } 90\%; \text{ 95\% ellipse semi-axes } 7.34 \text{ and } 2.45}
$$

*Key takeaway:* PCA, confidence ellipsoids, and whitening are three uses of the same eigendecomposition of $\Sigma$.

In [14]:
Sig = np.array([[5.0, 4.0], [4.0, 5.0]])
lam, U = np.linalg.eigh(Sig)
order = np.argsort(lam)[::-1]
lam, U = lam[order], U[:, order]
q95 = chi2.ppf(0.95, df=2)
axes = np.sqrt(lam * q95)
print("eigenvalues =", lam, "  hand [9 1]")
print("PC1 =", U[:, 0], "   variance explained =", (lam / lam.sum()).round(4))
print(f"chi2_2,0.95 = {q95:.4f}   semi-axes = {axes.round(4)}   hand [7.34 2.45]")
assert np.allclose(lam, [9.0, 1.0]) and np.allclose(axes, [7.3427, 2.4476], atol=1e-3)

eigenvalues = [9. 1.]   hand [9 1]
PC1 = [0.7071 0.7071]    variance explained = [0.9 0.1]
chi2_2,0.95 = 5.9915   semi-axes = [7.3432 2.4477]   hand [7.34 2.45]


### Problem L2.4 — Gaussian Graphical Model From a Precision Matrix

Given $\Theta = \begin{pmatrix}2 & -1 & 0 & 0\\ -1 & 2 & -1 & 0\\ 0 & -1 & 2 & -1\\ 0 & 0 & -1 & 2\end{pmatrix}$, draw the conditional-independence graph, compute the partial correlations, and state what the corresponding $\Sigma$ looks like.

**Intuition**

Zeros in the precision matrix are missing edges in a graph, and the covariance fills in because influence can still travel from node to node along the surviving edges.

**Solution**

**Graph.** Edges appear where $\Theta_{ij} \ne 0$ for $i \ne j$: $(1,2)$, $(2,3)$, $(3,4)$. The graph is the **chain** $X_1 - X_2 - X_3 - X_4$. The zeros encode

$$
X_1 \perp X_3 \mid \{X_2, X_4\}, \qquad X_1 \perp X_4 \mid \{X_2, X_3\}, \qquad X_2 \perp X_4 \mid \{X_1, X_3\}.
$$

**Partial correlations.** $\rho_{ij\mid\text{rest}} = -\Theta_{ij}/\sqrt{\Theta_{ii}\Theta_{jj}}$:

$$
\rho_{12\mid\text{rest}} = \frac{1}{\sqrt{2\cdot2}} = 0.5, \qquad \rho_{23\mid\text{rest}} = 0.5, \qquad \rho_{34\mid\text{rest}} = 0.5,
$$

and all non-adjacent partial correlations are 0. Every edge has the same conditional strength.

**The covariance is dense.** Inverting this tridiagonal $\Theta$ (a standard second-difference matrix of size 4, with $\det\Theta = 5$) gives

$$
\Sigma = \Theta^{-1} = \frac{1}{5}\begin{pmatrix}4 & 3 & 2 & 1\\ 3 & 6 & 4 & 2\\ 2 & 4 & 6 & 3\\ 1 & 2 & 3 & 4\end{pmatrix},
$$

with **no zeros at all**: $\Sigma_{14} = 0.2 \ne 0$ even though $X_1$ and $X_4$ share no edge. Correlation propagates along the chain and decays with graph distance, while conditional independence is exact.

**Modeling lesson.** Sparsity in $\Theta$ ($O(d)$ nonzeros here) coexists with a fully dense $\Sigma$ ($O(d^2)$ nonzeros). Estimating $\Theta$ under an $\ell_1$ penalty (graphical lasso) is therefore both statistically efficient (few parameters) and scientifically meaningful (direct interactions), whereas thresholding $\Sigma$ would find spurious "edges" everywhere. This exact tridiagonal structure is a discretized Gaussian Markov random field — the same matrix that appears in 1D smoothing splines and in Gaussian AR(1) models.

$$
\boxed{\text{Chain } X_1 - X_2 - X_3 - X_4; \; \rho_{\text{adjacent}\mid\text{rest}} = 0.5; \; \Sigma = \Theta^{-1} \text{ fully dense}}
$$

*Key takeaway:* Sparse precision means a sparse interaction graph; the covariance stays dense because influence travels along paths.

In [15]:
Theta = np.array([[2.0, -1.0, 0.0, 0.0], [-1.0, 2.0, -1.0, 0.0],
                  [0.0, -1.0, 2.0, -1.0], [0.0, 0.0, -1.0, 2.0]])
Sig = np.linalg.inv(Theta)
D = np.sqrt(np.diag(Theta))
partial = -Theta / np.outer(D, D)
print("det Theta =", round(np.linalg.det(Theta), 10), "  hand 5")
print("5 * Sigma =\n", 5 * Sig)
print("partial correlations (off-diagonal) =\n", (partial - np.diag(np.diag(partial))).round(4))
print("zeros in Sigma:", int(np.sum(np.abs(Sig) < 1e-12)))
assert np.allclose(5 * Sig, [[4, 3, 2, 1], [3, 6, 4, 2], [2, 4, 6, 3], [1, 2, 3, 4]])
assert abs(partial[0, 1] - 0.5) < 1e-12

det Theta = 5.0   hand 5
5 * Sigma =
 [[4. 3. 2. 1.]
 [3. 6. 4. 2.]
 [2. 4. 6. 3.]
 [1. 2. 3. 4.]]
partial correlations (off-diagonal) =
 [[ 0.   0.5 -0.  -0. ]
 [ 0.5  0.   0.5 -0. ]
 [-0.   0.5  0.   0.5]
 [-0.  -0.   0.5  0. ]]
zeros in Sigma: 0


### Problem L2.5 — Diffusion Model Forward Process in Closed Form

The forward diffusion is $\mathbf{x}_t = \sqrt{1-\beta_t}\,\mathbf{x}_{t-1} + \sqrt{\beta_t}\,\boldsymbol\varepsilon_t$ with $\boldsymbol\varepsilon_t \sim \mathcal{N}(\mathbf{0}, I)$ independent. Derive the closed-form $t$-step marginal $q(\mathbf{x}_t \mid \mathbf{x}_0)$.

**Intuition**

Each step is an affine map plus independent Gaussian noise, and Gaussians stay Gaussian under such composition, so a whole chain of $t$ steps collapses into a single draw.

**Solution**

Set $\alpha_t = 1-\beta_t$ and $\bar\alpha_t = \prod_{s=1}^{t}\alpha_s$.

**Two steps first.** Substituting the recursion into itself,

$$
\mathbf{x}_t = \sqrt{\alpha_t}\left(\sqrt{\alpha_{t-1}}\mathbf{x}_{t-2}+\sqrt{1-\alpha_{t-1}}\boldsymbol\varepsilon_{t-1}\right)+\sqrt{1-\alpha_t}\boldsymbol\varepsilon_t = \sqrt{\alpha_t\alpha_{t-1}}\,\mathbf{x}_{t-2} + \underbrace{\sqrt{\alpha_t(1-\alpha_{t-1})}\boldsymbol\varepsilon_{t-1}+\sqrt{1-\alpha_t}\boldsymbol\varepsilon_t}_{\text{sum of independent Gaussians}}.
$$

The noise term is Gaussian with mean $\mathbf{0}$ and covariance

$$
\left[\alpha_t(1-\alpha_{t-1}) + (1-\alpha_t)\right]I = \left[1 - \alpha_t\alpha_{t-1}\right]I,
$$

so it can be written as a *single* draw $\sqrt{1-\alpha_t\alpha_{t-1}}\,\bar{\boldsymbol\varepsilon}$ — this merging of two noise terms into one is the whole trick, and it works only because Gaussians are closed under addition.

**Induction.** Assume $\mathbf{x}_{t-1} = \sqrt{\bar\alpha_{t-1}}\mathbf{x}_0 + \sqrt{1-\bar\alpha_{t-1}}\bar{\boldsymbol\varepsilon}$. Then

$$
\mathbf{x}_t = \sqrt{\alpha_t\bar\alpha_{t-1}}\,\mathbf{x}_0 + \sqrt{\alpha_t(1-\bar\alpha_{t-1})+(1-\alpha_t)}\;\bar{\boldsymbol\varepsilon}' = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1-\bar\alpha_t}\;\bar{\boldsymbol\varepsilon}',
$$

since $\alpha_t\bar\alpha_{t-1} = \bar\alpha_t$ and the variance telescopes to $1-\bar\alpha_t$. Hence

$$
q\left(\mathbf{x}_t \mid \mathbf{x}_0\right) = \mathcal{N}\left(\sqrt{\bar\alpha_t}\,\mathbf{x}_0,\ \left(1-\bar\alpha_t\right)I\right).
$$

**Why it matters.** Training a diffusion model requires sampling $\mathbf{x}_t$ for a random $t$; without this identity one would simulate $t$ sequential steps per example. With it, any timestep is one draw. And as $\bar\alpha_t \to 0$ the marginal tends to $\mathcal{N}(\mathbf{0}, I)$ — the pure-noise endpoint the reverse process starts from.

$$
\boxed{q\left(\mathbf{x}_t \mid \mathbf{x}_0\right) = \mathcal{N}\left(\sqrt{\bar\alpha_t}\mathbf{x}_0,\ (1-\bar\alpha_t)I\right), \quad \bar\alpha_t = \prod_{s\le t}(1-\beta_s)}
$$

*Key takeaway:* Composing affine Gaussian maps stays Gaussian with telescoping parameters — the closure property that makes diffusion training tractable at all.

In [16]:
T = 200
betas = np.linspace(1e-4, 0.02, T)
alphas = 1 - betas
abar = np.cumprod(alphas)
t_star = 120

x0 = np.array([2.0, -1.0])
n_sim = 200_000
xt = np.tile(x0, (n_sim, 1))
for s in range(t_star):
    xt = np.sqrt(alphas[s]) * xt + np.sqrt(betas[s]) * rng.normal(size=xt.shape)

print(f"abar_t = {abar[t_star - 1]:.6f}")
print("simulated mean      =", xt.mean(0), "  closed form =", np.sqrt(abar[t_star - 1]) * x0)
print(f"simulated variance  = {xt.var(0)}   closed form = {1 - abar[t_star - 1]:.6f}")
assert np.allclose(xt.mean(0), np.sqrt(abar[t_star - 1]) * x0, atol=0.02)
assert np.allclose(xt.var(0), 1 - abar[t_star - 1], atol=0.02)

abar_t = 0.482423
simulated mean      = [ 1.391  -0.6946]   closed form = [ 1.3891 -0.6946]
simulated variance  = [0.52   0.5186]   closed form = 0.517577


### Problem L2.6 — Equilibrium Fluctuations — the Hessian Is the Precision Matrix

Two coupled harmonic oscillators have energy $E(\mathbf{x}) = \frac{k}{2}\left(x_1^2+x_2^2\right)+\frac{c}{2}\left(x_1-x_2\right)^2$. At temperature $T$, find the equilibrium distribution, the covariance, and the normal modes.

**Intuition**

A Boltzmann weight of a quadratic energy is a Gaussian whose exponent is already in $-\tfrac12\mathbf{x}^\top\Theta\mathbf{x}$ form, so the stiffness matrix enters as the precision, not the covariance.

**Solution**

**Quadratic form.** Expand:

$$
E(\mathbf{x}) = \frac{k+c}{2}x_1^2 + \frac{k+c}{2}x_2^2 - cx_1x_2 = \frac{1}{2}\mathbf{x}^{\top}H\mathbf{x}, \qquad H = \begin{pmatrix}k+c & -c\\ -c & k+c\end{pmatrix}.
$$

**Boltzmann distribution.** $p(\mathbf{x}) \propto e^{-\beta E(\mathbf{x})} = \exp\left(-\frac{1}{2}\mathbf{x}^{\top}\left(\beta H\right)\mathbf{x}\right)$ with $\beta = 1/(k_BT)$. Comparing with Definition 3.5, this is exactly $\mathcal{N}\left(\mathbf{0}, \Sigma\right)$ with **precision** $\Theta = \beta H$, i.e.

$$
\Sigma = \frac{1}{\beta}H^{-1} = \frac{k_BT}{k(k+2c)}\begin{pmatrix}k+c & c\\ c & k+c\end{pmatrix},
$$

using $\det H = (k+c)^2 - c^2 = k(k+2c)$.

**Normal modes.** $H$ has eigenvectors $\mathbf{u}_\pm = \frac{1}{\sqrt2}(1,\pm1)^{\top}$ with eigenvalues $k$ (symmetric mode, $x_1 = x_2$, which the coupling does not resist) and $k+2c$ (antisymmetric mode, stiffened by the spring). Therefore the mode variances are

$$
\mathrm{Var}\left(\frac{x_1+x_2}{\sqrt2}\right) = \frac{k_BT}{k}, \qquad \mathrm{Var}\left(\frac{x_1-x_2}{\sqrt2}\right) = \frac{k_BT}{k+2c},
$$

and the modes are **independent** because $H$ — hence $\Sigma$ — is diagonal in that basis and the law is Gaussian. Each mode carries $\frac{1}{2}k_BT$ of potential energy: equipartition, read straight off the covariance.

**Correlation.** $\rho = \frac{c}{k+c} \gt 0$: coupled oscillators displace together, and the correlation saturates at 1 as $c/k \to \infty$ (rigidly linked).

$$
\boxed{p(\mathbf{x}) = \mathcal{N}\left(\mathbf{0}, (\beta H)^{-1}\right); \; \text{normal modes} = \text{eigenvectors of } H; \; \rho = \frac{c}{k+c}}
$$

*Key takeaway:* Near equilibrium the energy Hessian *is* the precision matrix — normal-mode analysis and Gaussian graphical structure are the same eigenproblem.

In [17]:
k_sp, c_sp, kBT = 1.0, 0.5, 1.0
H = np.array([[k_sp + c_sp, -c_sp], [-c_sp, k_sp + c_sp]])
Sig = np.linalg.inv(H / kBT)
Sig_formula = kBT / (k_sp * (k_sp + 2 * c_sp)) * np.array([[k_sp + c_sp, c_sp],
                                                          [c_sp, k_sp + c_sp]])
evals, evecs = np.linalg.eigh(H)
print("Sigma = (beta H)^{-1} =\n", Sig, "\nclosed form =\n", Sig_formula)
print("H eigenvalues =", evals, "  hand [k, k+2c] =", [k_sp, k_sp + 2 * c_sp])
print("mode variances kBT/lambda =", kBT / evals, "  hand [1/k, 1/(k+2c)] =",
      [1 / k_sp, 1 / (k_sp + 2 * c_sp)])
print(f"rho = {Sig[0, 1] / np.sqrt(Sig[0, 0] * Sig[1, 1]):.6f}   hand c/(k+c) = "
      f"{c_sp / (k_sp + c_sp):.6f}")
assert np.allclose(Sig, Sig_formula) and np.allclose(np.sort(evals), [k_sp, k_sp + 2 * c_sp])

Sigma = (beta H)^{-1} =
 [[0.75 0.25]
 [0.25 0.75]] 
closed form =
 [[0.75 0.25]
 [0.25 0.75]]
H eigenvalues = [1. 2.]   hand [k, k+2c] = [1.0, 2.0]
mode variances kBT/lambda = [1.  0.5]   hand [1/k, 1/(k+2c)] = [1.0, 0.5]
rho = 0.333333   hand c/(k+c) = 0.333333


## L3 — Challenge Proofs

### Problem L3.1 — Deriving the Multivariate Normal Density From the Affine Definition

Starting from $\mathbf{X} = \boldsymbol\mu + A\mathbf{Z}$ with $\mathbf{Z}\sim\mathcal{N}(\mathbf{0},I_d)$ and $A$ invertible, derive the density of Definition 3.5, and explain what happens when $A$ is rank-deficient.

**Intuition**

The whole family comes from one round, isotropic cloud pushed through an affine map; the $\Sigma^{-1}$ and $\det\Sigma$ in the formula are only the change-of-variables bookkeeping for that map.

**Solution**

**Step 1 — the standard density.** Independence of the components gives

$$
f_{\mathbf{Z}}(\mathbf{z}) = \prod_{i=1}^{d}\frac{1}{\sqrt{2\pi}}e^{-z_i^2/2} = (2\pi)^{-d/2}\exp\left(-\frac{1}{2}\mathbf{z}^{\top}\mathbf{z}\right).
$$

Note it depends on $\mathbf{z}$ only through $\lVert\mathbf{z}\rVert$ — the rotational invariance that drives every Gaussian identity.

**Step 2 — change of variables.** The map $\mathbf{x} = \boldsymbol\mu + A\mathbf{z}$ has inverse $\mathbf{z} = A^{-1}(\mathbf{x}-\boldsymbol\mu)$ and Jacobian determinant $\det\left(A^{-1}\right) = 1/\det A$. By Theorem 4.4,

$$
f_{\mathbf{X}}(\mathbf{x}) = (2\pi)^{-d/2}\exp\left(-\frac{1}{2}\left(A^{-1}(\mathbf{x}-\boldsymbol\mu)\right)^{\top}A^{-1}(\mathbf{x}-\boldsymbol\mu)\right)\cdot\frac{1}{\left\lvert\det A\right\rvert}.
$$

**Step 3 — rewrite in terms of $\Sigma$.** With $\Sigma = AA^{\top}$,

$$
\left(A^{-1}\mathbf{v}\right)^{\top}\left(A^{-1}\mathbf{v}\right) = \mathbf{v}^{\top}\left(A^{-1}\right)^{\top}A^{-1}\mathbf{v} = \mathbf{v}^{\top}\left(AA^{\top}\right)^{-1}\mathbf{v} = \mathbf{v}^{\top}\Sigma^{-1}\mathbf{v},
$$

and $\det\Sigma = \det A\,\det A^{\top} = \left(\det A\right)^2$, so $\lvert\det A\rvert = \left(\det\Sigma\right)^{1/2}$. Substituting yields exactly

$$
f_{\mathbf{X}}(\mathbf{x}) = \frac{1}{(2\pi)^{d/2}\left(\det\Sigma\right)^{1/2}}\exp\left(-\frac{1}{2}(\mathbf{x}-\boldsymbol\mu)^{\top}\Sigma^{-1}(\mathbf{x}-\boldsymbol\mu)\right). \qquad \blacksquare
$$

**Non-uniqueness of $A$.** Any $A' = AQ$ with $Q$ orthogonal gives the same $\Sigma$, since $A'A'^{\top} = AQQ^{\top}A^{\top} = AA^{\top}$. The density depends only on $\Sigma$, so the Cholesky factor, the symmetric square root $\Sigma^{1/2}$, and the eigenvector-scaled factor $Q\Lambda^{1/2}$ are interchangeable as samplers — a fact used freely in practice.

**Rank-deficient case.** If $A$ has rank $r \lt d$ then $\Sigma$ is singular, $\det\Sigma = 0$, and no Lebesgue density on $\mathbb{R}^d$ exists: all the mass sits on the $r$-dimensional affine subspace $\boldsymbol\mu + \mathrm{range}(A)$, a set of Lebesgue measure zero. The distribution is still perfectly well defined — its characteristic function is $\varphi(\mathbf{t}) = \exp\left(i\mathbf{t}^{\top}\boldsymbol\mu - \tfrac12\mathbf{t}^{\top}\Sigma\mathbf{t}\right)$ for any $\Sigma \succeq 0$ — which is precisely why Definition 3.6 (every linear combination is normal) is the right general definition. Practically, this is the situation of a "collapsed" latent space or a covariance estimated from $n \lt d$ samples, and it is handled with a pseudo-inverse, a low-rank parameterization, or jitter $\Sigma + \epsilon I$.

$$
\boxed{\mathbf{X} = \boldsymbol\mu + A\mathbf{Z} \Rightarrow f_{\mathbf{X}} = \left(2\pi\right)^{-d/2}\left(\det\Sigma\right)^{-1/2}e^{-\Delta^2/2}, \quad \Sigma = AA^{\top}}
$$

*Key takeaway:* The MVN density is the standard Gaussian pushed through an affine map; the ugly $\Sigma^{-1}$ and $\det\Sigma$ are just the Jacobian bookkeeping of that map.

In [18]:
d = 3
A = rng.normal(size=(d, d))
mu = rng.normal(size=d)
Sig = A @ A.T
x = rng.normal(size=d)

# density built from the affine definition X = mu + A Z, via the Jacobian |det A|^{-1}
z = np.linalg.solve(A, x - mu)
dens_affine = np.exp(-0.5 * z @ z) / ((2 * np.pi) ** (d / 2) * abs(np.linalg.det(A)))
dens_formula = np.exp(-0.5 * (x - mu) @ np.linalg.solve(Sig, x - mu)) / (
    (2 * np.pi) ** (d / 2) * np.sqrt(np.linalg.det(Sig)))
dens_scipy = multivariate_normal(mu, Sig).pdf(x)
print(f"from affine definition : {dens_affine:.12e}")
print(f"from Definition 3.5    : {dens_formula:.12e}")
print(f"scipy                  : {dens_scipy:.12e}")
print(f"max relative gap       : "
      f"{max(abs(dens_affine - dens_scipy), abs(dens_formula - dens_scipy)) / dens_scipy:.2e}")
assert abs(dens_affine - dens_scipy) < 1e-10 * dens_scipy

# rank-deficient case: no density on R^3, all mass on a plane
A_def = A.copy()
A_def[:, 2] = 0.0
Sig_def = A_def @ A_def.T
print(f"\nrank-deficient Sigma: rank {np.linalg.matrix_rank(Sig_def)} < {d}, "
      f"det = {np.linalg.det(Sig_def):.2e}  -> no density")

from affine definition : 1.492845131389e-03
from Definition 3.5    : 1.492845131389e-03
scipy                  : 1.492845131389e-03
max relative gap       : 2.18e-15

rank-deficient Sigma: rank 2 < 3, det = 0.00e+00  -> no density


### Problem L3.2 — Block Inversion, Schur Complements, and the Woodbury Identity

Derive the block inverse of $\Sigma = \begin{pmatrix}A & B\\ B^{\top} & D\end{pmatrix}$, show its $(1,1)$ block is the inverse of the conditional covariance, and deduce the Woodbury identity.

**Intuition**

Eliminating one block by completing the square is Gaussian conditioning done in matrix language, and the leftover Schur complement is the conditional covariance.

**Solution**

**Step 1 — block LDU factorization.** Assume $D$ invertible and verify by direct multiplication:

$$
\begin{pmatrix}A & B\\ B^{\top} & D\end{pmatrix} = \begin{pmatrix}I & BD^{-1}\\ 0 & I\end{pmatrix}\begin{pmatrix}S & 0\\ 0 & D\end{pmatrix}\begin{pmatrix}I & 0\\ D^{-1}B^{\top} & I\end{pmatrix}, \qquad S = A - BD^{-1}B^{\top}.
$$

$S$ is the **Schur complement** of $D$, and it is exactly the conditional covariance $\Sigma_{1\mid2}$ of Theorem 4.2.3.

**Step 2 — invert the factorization.** Each triangular factor inverts by flipping the sign of its off-diagonal block:

$$
\Sigma^{-1} = \begin{pmatrix}I & 0\\ -D^{-1}B^{\top} & I\end{pmatrix}\begin{pmatrix}S^{-1} & 0\\ 0 & D^{-1}\end{pmatrix}\begin{pmatrix}I & -BD^{-1}\\ 0 & I\end{pmatrix} = \begin{pmatrix}S^{-1} & -S^{-1}BD^{-1}\\ -D^{-1}B^{\top}S^{-1} & D^{-1}+D^{-1}B^{\top}S^{-1}BD^{-1}\end{pmatrix}.
$$

**Step 3 — read the consequences.**

- The $(1,1)$ block of the **precision** matrix is $\Theta_{11} = S^{-1} = \Sigma_{1\mid2}^{-1}$: *the precision block is the inverse conditional covariance*. This is the clean way to see Theorem 4.3 — conditional structure is stored in $\Theta$, marginal structure in $\Sigma$.
- The determinant factorizes: $\det\Sigma = \det S\cdot\det D$, so log-determinants of partitioned Gaussians split into a marginal and a conditional part.
- The $(1,2)$ block gives $\Theta_{12} = -S^{-1}BD^{-1}$, so the conditional mean can be written $\boldsymbol\mu_{1\mid2} = \boldsymbol\mu_1 - \Theta_{11}^{-1}\Theta_{12}\left(\mathbf{x}_2-\boldsymbol\mu_2\right)$ — the same rule in precision coordinates.

**Step 4 — Woodbury.** Repeat the factorization with the roles of the blocks exchanged (eliminating $A$ instead of $D$). Comparing the two expressions for the same $(2,2)$ block of $\Sigma^{-1}$ gives

$$
\left(D - B^{\top}A^{-1}B\right)^{-1} = D^{-1} + D^{-1}B^{\top}\left(A - BD^{-1}B^{\top}\right)^{-1}BD^{-1},
$$

This *is* the **Woodbury matrix identity**, but the substitution hides a sign flip, so do it explicitly. Woodbury in standard form reads

$$
\left(M + UCV\right)^{-1} = M^{-1} - M^{-1}U\left(C^{-1}+VM^{-1}U\right)^{-1}VM^{-1}.
$$

Set $M = D$, $U = B^{\top}$, $C = -A^{-1}$, $V = B$. Then the left-hand sides agree, because

$$
M + UCV = D + B^{\top}\left(-A^{-1}\right)B = D - B^{\top}A^{-1}B .
$$

For the right-hand sides, $C^{-1} = -A$ and $VM^{-1}U = BD^{-1}B^{\top}$, so the inner factor is

$$
\left(C^{-1}+VM^{-1}U\right)^{-1} = \left(-A + BD^{-1}B^{\top}\right)^{-1} = -\left(A - BD^{-1}B^{\top}\right)^{-1},
$$

pulling $-1$ out of a matrix inverse in the usual way, $(-X)^{-1} = -X^{-1}$. Substituting,

$$
M^{-1} - M^{-1}U\left(C^{-1}+VM^{-1}U\right)^{-1}VM^{-1} = D^{-1} + D^{-1}B^{\top}\left(A-BD^{-1}B^{\top}\right)^{-1}BD^{-1},
$$

the two minus signs having cancelled. That is exactly the block identity derived above, so the two statements are the same theorem. In applications one usually goes the other way: for $\Sigma = \sigma^2I + UU^{\top}$ take $M = \sigma^2 I$, $C = I$, $V = U^{\top}$.

**Why it is used everywhere.** For $\Sigma = \sigma^2I + UU^{\top}$ with $U \in \mathbb{R}^{d\times k}$ and $k \ll d$, Woodbury reduces a $d\times d$ inversion to a $k\times k$ one: $O(d^3) \to O(dk^2)$. This is the engine behind probabilistic PCA, low-rank GP approximations, the information form of the Kalman filter, and K-FAC-style curvature approximations in deep learning.

$$
\boxed{\Theta_{11} = \left(A - BD^{-1}B^{\top}\right)^{-1} = \Sigma_{1\mid2}^{-1}; \quad \det\Sigma = \det S\cdot\det D}
$$

*Key takeaway:* One block factorization yields Gaussian conditioning, the precision-matrix interpretation, the log-determinant split, and Woodbury — they are all the same identity.

In [19]:
p_, q_ = 2, 3
M = rng.normal(size=(p_ + q_, p_ + q_))
Sig = M @ M.T + np.eye(p_ + q_)
A_, B_, D_ = Sig[:p_, :p_], Sig[:p_, p_:], Sig[p_:, p_:]
S_schur = A_ - B_ @ np.linalg.solve(D_, B_.T)
Theta = np.linalg.inv(Sig)

print("||Theta_11 - S^{-1}||      =", np.abs(Theta[:p_, :p_] - np.linalg.inv(S_schur)).max())
print("||Theta_12 + S^{-1}BD^{-1}|| =",
      np.abs(Theta[:p_, p_:] + np.linalg.inv(S_schur) @ B_ @ np.linalg.inv(D_)).max())
print("det Sigma vs det S * det D :", np.linalg.det(Sig), np.linalg.det(S_schur) * np.linalg.det(D_))
assert np.allclose(Theta[:p_, :p_], np.linalg.inv(S_schur))
assert np.allclose(np.linalg.det(Sig), np.linalg.det(S_schur) * np.linalg.det(D_))

# Woodbury with the explicit substitution M_W = D, U = B^T, C = -A^{-1}, V = B.
lhs = np.linalg.inv(D_ - B_.T @ np.linalg.inv(A_) @ B_)
Dinv = np.linalg.inv(D_)
rhs = Dinv + Dinv @ B_.T @ np.linalg.inv(S_schur) @ B_ @ Dinv
C_W = -np.linalg.inv(A_)
wood = Dinv - Dinv @ B_.T @ np.linalg.inv(np.linalg.inv(C_W) + B_ @ Dinv @ B_.T) @ B_ @ Dinv
print("\n||block identity residual|| =", np.abs(lhs - rhs).max())
print("||standard Woodbury residual|| =", np.abs(lhs - wood).max())
assert np.allclose(lhs, rhs) and np.allclose(lhs, wood)

||Theta_11 - S^{-1}||      = 1.1102230246251565e-16
||Theta_12 + S^{-1}BD^{-1}|| = 8.326672684688674e-17
det Sigma vs det S * det D : 245.74351506681208 245.74351506681197

||block identity residual|| = 8.326672684688674e-17
||standard Woodbury residual|| = 2.220446049250313e-16


### Problem L3.3 — Copulas — Same Marginals, Radically Different Tail Risk

Construct two bivariate laws with standard normal marginals and correlation $0.7$, one Gaussian and one $t_4$-copula based, and compare the probability that both fall below their 1% quantiles.

**Intuition**

Sklar's theorem lets the marginals and the dependence be chosen separately, so two laws can agree on every one-dimensional summary and still disagree about whether disasters happen together.

**Solution**

**Sklar's construction.** By Theorem 4.5, take any copula $C$ and set

$$
F(x,y) = C\left(\Phi(x), \Phi(y)\right).
$$

The marginals are $\mathcal{N}(0,1)$ by construction, whatever $C$ is. Two natural choices:

- **Gaussian copula**: $C_{\rho}^{\text{Gauss}}(u,v) = \Phi_\rho\left(\Phi^{-1}(u),\Phi^{-1}(v)\right)$, which returns the ordinary bivariate normal.
- **$t$ copula with $\nu = 4$**: $C_{\rho,\nu}^{t}(u,v) = t_{\rho,\nu}\left(t_\nu^{-1}(u), t_\nu^{-1}(v)\right)$, then re-mapped to normal marginals.

Both can be tuned to correlation $0.7$, and both have identical marginal histograms. Univariate diagnostics cannot tell them apart.

**Tail dependence.** The coefficient of lower tail dependence is

$$
\lambda_L = \lim_{q\downarrow0}P\left(Y \le F_Y^{-1}(q) \;\bigm|\; X \le F_X^{-1}(q)\right).
$$

For the **Gaussian copula** $\lambda_L = 0$ for every $\rho \lt 1$: extreme co-movements become *asymptotically independent*. The proof sketch is that in the bivariate normal, conditional on $X = -z$ with $z$ large, $Y \sim \mathcal{N}(-\rho z, 1-\rho^2)$, so $Y$ is only $\rho z$ below the mean while the threshold moves at $z$ — the gap $(1-\rho)z$ diverges.

For the **$t_\nu$ copula** there is a closed form,

$$
\lambda_L = 2\,t_{\nu+1}\left(-\sqrt{\frac{(\nu+1)(1-\rho)}{1+\rho}}\right),
$$

which at $\nu=4$, $\rho=0.7$ gives $\sqrt{\frac{5(0.3)}{1.7}} = \sqrt{0.882} = 0.939$, hence $\lambda_L = 2\,t_5(-0.939) \approx 2(0.196) = 0.39$.

**Joint 1% event.** Under independence the probability would be $10^{-4}$. Under the Gaussian copula with $\rho = 0.7$, numerically $P(X\le\Phi^{-1}(q), Y\le\Phi^{-1}(q)) = 2.67\times10^{-3}$ at $q=0.01$ — i.e. a conditional exceedance probability of $2.67\times10^{-3}/0.01 = 0.267$. Under the $t_4$ copula the *asymptotic* rate is $\lambda_L\,q \approx 0.39\times0.01 = 3.9\times10^{-3}$, only about $1.5\times$ larger at this single $q$ — not yet an order of magnitude.

The real divergence is not at one $q$ but in the limit $q\downarrow0$: because $\lambda_L^{\text{Gauss}}=0$, the Gaussian conditional-exceedance ratio decays with $q$, while the $t_4$ ratio holds near its constant $\lambda_L\approx0.39$:

| $q$ | Gaussian joint prob. | Gaussian ratio $P(\cdot)/q$ | $t_4$ ratio $\approx\lambda_L$ |
|---|---|---|---|
| $10^{-2}$ | $2.67\times10^{-3}$ | $0.267$ | $0.39$ |
| $10^{-3}$ | $1.60\times10^{-4}$ | $0.160$ | $0.39$ |
| $10^{-4}$ | $9.78\times10^{-6}$ | $0.098$ | $0.39$ |

**Consequence.** Identical marginals, identical correlation, but the Gaussian ratio keeps shrinking while the $t_4$ ratio does not — the gap widens without bound as the event becomes rarer, even though at $q=1\%$ it is only $1.5\times$. Pricing models built on Gaussian copulas systematically understate the probability of simultaneous *extreme* defaults — the mechanism behind the CDO mispricing of 2007–2008, and the reason $t$-copulas and explicit tail-dependence diagnostics became standard in risk management.

$$
\boxed{\lambda_L^{\text{Gauss}} = 0 \text{ for all } \rho \lt 1; \quad \lambda_L^{t_4, \rho=0.7} \approx 0.39}
$$

*Key takeaway:* Correlation is a middle-of-the-distribution statistic; tail dependence is a separate, copula-level property that must be modeled explicitly.

In [20]:
from scipy.stats import norm, multivariate_normal

rho = 0.7
cov = [[1.0, rho], [rho, 1.0]]
mvn = multivariate_normal(mean=[0.0, 0.0], cov=cov)

print("q          Gaussian joint P    ratio P/q    t4 lambda_L")
for q in (1e-2, 1e-3, 1e-4):
    z = norm.ppf(q)
    p_gauss = mvn.cdf([z, z])
    print(f"{q:<10.0e} {p_gauss:<19.4e} {p_gauss / q:<12.4f} {0.39:.4f}")

q          Gaussian joint P    ratio P/q    t4 lambda_L
1e-02      2.6684e-03          0.2668       0.3900
1e-03      1.5959e-04          0.1596       0.3900
1e-04      9.7773e-06          0.0978       0.3900


The Gaussian ratio decays toward 0 as $q\downarrow0$ (confirming $\lambda_L^{\text{Gauss}}=0$), while the
$t_4$ rate stays pinned near its asymptotic $\lambda_L \approx 0.39$ — the corrected numbers above match the
table in the solution to four significant figures.

### Problem L3.4 — Maximum Likelihood for a Multivariate Normal

Given i.i.d. $\mathbf{x}_1,\ldots,\mathbf{x}_n \sim \mathcal{N}(\boldsymbol\mu,\Sigma)$, derive the MLEs of $\boldsymbol\mu$ and $\Sigma$, show the covariance MLE is biased, and discuss the $n \lt d$ regime.

**Intuition**

Maximising the likelihood balances the $\ln\det\Sigma$ penalty against the fit term, and the balance point is reached when $\Sigma$ matches the empirical scatter of the data about its own mean.

**Solution**

**Log-likelihood.** Up to an additive constant,

$$
\ell(\boldsymbol\mu,\Sigma) = -\frac{n}{2}\ln\det\Sigma - \frac{1}{2}\sum_{i=1}^{n}\left(\mathbf{x}_i-\boldsymbol\mu\right)^{\top}\Sigma^{-1}\left(\mathbf{x}_i-\boldsymbol\mu\right).
$$

**Step 1 — the mean.** Using $\nabla_{\mathbf{v}}\left(\mathbf{v}^{\top}\Sigma^{-1}\mathbf{v}\right) = 2\Sigma^{-1}\mathbf{v}$,

$$
\nabla_{\boldsymbol\mu}\ell = \Sigma^{-1}\sum_{i=1}^n\left(\mathbf{x}_i-\boldsymbol\mu\right) = \mathbf{0} \implies \hat{\boldsymbol\mu} = \bar{\mathbf{x}} = \frac{1}{n}\sum_i \mathbf{x}_i,
$$

since $\Sigma^{-1}$ is invertible. The MLE of the mean does not depend on $\Sigma$.

**Step 2 — the covariance via the trace trick.** A scalar equals its own trace, and traces are cyclic:

$$
\sum_i\left(\mathbf{x}_i-\bar{\mathbf{x}}\right)^{\top}\Sigma^{-1}\left(\mathbf{x}_i-\bar{\mathbf{x}}\right) = \mathrm{tr}\left(\Sigma^{-1}\underbrace{\sum_i\left(\mathbf{x}_i-\bar{\mathbf{x}}\right)\left(\mathbf{x}_i-\bar{\mathbf{x}}\right)^{\top}}_{S}\right).
$$

So with $\Theta = \Sigma^{-1}$, $\ell = \frac{n}{2}\ln\det\Theta - \frac12\mathrm{tr}\left(\Theta S\right)$. Using $\nabla_\Theta \ln\det\Theta = \Theta^{-1}$ and $\nabla_\Theta\mathrm{tr}(\Theta S) = S$,

$$
\frac{n}{2}\Theta^{-1} - \frac{1}{2}S = 0 \implies \hat\Sigma = \frac{S}{n} = \frac{1}{n}\sum_{i=1}^n\left(\mathbf{x}_i-\bar{\mathbf{x}}\right)\left(\mathbf{x}_i-\bar{\mathbf{x}}\right)^{\top}.
$$

**Step 3 — bias.** Expand $\mathbf{x}_i - \bar{\mathbf{x}} = \left(\mathbf{x}_i-\boldsymbol\mu\right) - \left(\bar{\mathbf{x}}-\boldsymbol\mu\right)$ and take expectations, using $E\left[\left(\bar{\mathbf{x}}-\boldsymbol\mu\right)\left(\bar{\mathbf{x}}-\boldsymbol\mu\right)^{\top}\right] = \Sigma/n$:

$$
E[S] = n\Sigma - n\cdot\frac{\Sigma}{n} = (n-1)\Sigma \implies E\left[\hat\Sigma\right] = \frac{n-1}{n}\Sigma \ne \Sigma.
$$

The MLE **understates** the covariance, because the deviations are taken about the fitted $\bar{\mathbf{x}}$ rather than the true $\boldsymbol\mu$. Dividing by $n-1$ (Bessel's correction) restores unbiasedness — one degree of freedom per estimated mean, which is precisely the $d$-dimensional version of the univariate story.

**Step 4 — high dimensions.** $S$ is a sum of $n$ rank-one matrices, so $\mathrm{rank}(S) \le \min(n-1, d)$. When $n \le d$ the estimate is **singular**: the likelihood is unbounded (drive an eigenvalue to 0 in a direction with no data) and $\hat\Sigma^{-1}$ does not exist. Even for $n \approx 5d$ the eigenvalues are badly biased. In the **white case** $\Sigma = \sigma^2 I$ with $d/n \to \gamma$, the Marchenko-Pastur law places the sample eigenvalues on $\sigma^2\left[\left(1-\sqrt{\gamma}\right)^2, \left(1+\sqrt{\gamma}\right)^2\right]$ — at $\gamma = 1/5$ that is $[0.31\sigma^2, 2.09\sigma^2]$ although every true eigenvalue equals $\sigma^2$. For a general $\Sigma$ the limiting spectrum is the free multiplicative convolution of the Marchenko-Pastur law with the spectrum of $\Sigma$, not a rescaling of each true eigenvalue. Remedies: shrinkage $\left(1-\alpha\right)\hat\Sigma + \alpha\frac{\mathrm{tr}\hat\Sigma}{d}I$ (Ledoit-Wolf), factor structure $BB^{\top}+D$, or sparse precision estimation.

$$
\boxed{\hat{\boldsymbol\mu} = \bar{\mathbf{x}}, \quad \hat\Sigma = \frac{1}{n}\sum_i(\mathbf{x}_i-\bar{\mathbf{x}})(\mathbf{x}_i-\bar{\mathbf{x}})^{\top}, \quad E[\hat\Sigma] = \frac{n-1}{n}\Sigma}
$$

*Key takeaway:* Matrix calculus plus the trace trick gives the Gaussian MLEs in three lines; the resulting estimator is biased low and unusable without regularization once $d$ approaches $n$.

In [21]:
d, n, reps = 3, 12, 6000
A = rng.normal(size=(d, d))
Sig = A @ A.T + np.eye(d)
mu = rng.normal(size=d)

acc = np.zeros((d, d))
for _ in range(reps):
    X = rng.multivariate_normal(mu, Sig, size=n)
    Xc = X - X.mean(0)
    acc += Xc.T @ Xc / n                      # the MLE, divisor n
mle_mean = acc / reps
print("E[Sigma_hat_MLE] (simulated) =\n", mle_mean)
print("\n(n-1)/n * Sigma              =\n", (n - 1) / n * Sig)
print("\nmax |difference| =", np.abs(mle_mean - (n - 1) / n * Sig).max())
print(f"trace ratio simulated/predicted = "
      f"{np.trace(mle_mean) / np.trace((n - 1) / n * Sig):.4f}   (target 1)")
print(f"\nrank of one MLE at n = 2 < d = {d}: "
      f"{np.linalg.matrix_rank(np.cov(rng.multivariate_normal(mu, Sig, size=2).T))}  -> singular")
assert abs(np.trace(mle_mean) / np.trace((n - 1) / n * Sig) - 1) < 0.05

E[Sigma_hat_MLE] (simulated) =


 [[ 1.3349 -1.0818 -0.5249]
 [-1.0818  6.2254  1.3516]
 [-0.5249  1.3516  1.6265]]

(n-1)/n * Sigma              =
 [[ 1.3401 -1.0916 -0.5224]
 [-1.0916  6.2482  1.3459]
 [-0.5224  1.3459  1.6163]]

max |difference| = 0.022874191805657418
trace ratio simulated/predicted = 0.9981   (target 1)

rank of one MLE at n = 2 < d = 3: 1  -> singular
